# ST-02bis — Filtrage de la base SIRENE pour la siretisation

À partir des résultats de la sirenisation (déjà disponibles), on construit un
**échantillon réduit de SIRENE Etab** qui ne contient que les établissements
appartenant aux SIREN "intéressants" :

- SIREN validés en sirenisation (P1 + P2 + P3, statut VALIDE_FORT ou VALIDE)
- SIREN candidats top 5 en SN-P2 (même non validés)
- SIREN candidats top 3 en SN-P3 (même non validés)

Ces SIREN constituent un large ensemble de candidats plausibles. On garde
tous les établissements rattachés à ces SIREN pour le matching en ST-P2/P3.

**Pourquoi ?** La base SIRENE complète contient ~16M d'établissements,
ce qui rend les Phases 2 et 3 siretisation extrêmement lentes. En filtrant
sur les SIREN connus du périmètre FINESS, on réduit drastiquement le nombre
de candidats à scorer.

**Note** : ST-P1 (matching SIRET exact) reste sur la base complète.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.display     import afficher_tableau
from config.settings import (
    SIRENE_ETAB_CLEAN, SIRENE_ETAB_FILTRE,
    SN_PHASE1, SN_PHASE2, SN_PHASE3, PROCESSED_DIR,
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Récupération des SIREN cibles depuis la sirenisation

In [ ]:
import re

def _norm_siren(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return ""
    return re.sub(r"\s", "", str(val).strip())

sirens_cibles = set()

# Source 1 : SIREN validés dans les fichiers de PHASES de la sirenisation
def _collecter_sirens_valides(chemin, colonnes_siren):
    """Lit un fichier Excel, garde les feuilles Valide_fort + Valide,
    et collecte les SIREN depuis les colonnes données."""
    sirens = set()
    sheets = pd.read_excel(chemin, sheet_name=None, dtype=str)
    for nom in ('Valide_fort', 'Valide'):
        if nom in sheets:
            sdf = sheets[nom]
            for col in colonnes_siren:
                if col in sdf.columns:
                    vals = {_norm_siren(v) for v in sdf[col].dropna()}
                    vals.discard("")
                    sirens |= vals
    return sirens

# SN-P1 : colonne siren_ul (matching SIRET exact)
sirens_p1 = _collecter_sirens_valides(SN_PHASE1, ['siren_ul', 'nmsiren_stru'])
sirens_cibles |= sirens_p1
print(f"SIREN validés en SN-P1 : {len(sirens_p1):,}")

# SN-P2 : feuille Top5, on prend uniquement le rang 1 validé
df_p2 = pd.read_excel(SN_PHASE2, sheet_name='Top5', dtype=str)
df_p2['rang'] = pd.to_numeric(df_p2['rang'], errors='coerce')
r1_p2 = df_p2[df_p2['rang'] == 1]
v_p2 = r1_p2[r1_p2['statut_candidat'].isin(['VALIDE_FORT', 'VALIDE'])]
sirens_v_p2 = {_norm_siren(v) for v in v_p2.get('siren_ref', pd.Series(dtype=str)).dropna()}
sirens_v_p2.discard("")
sirens_cibles |= sirens_v_p2
print(f"SIREN validés en SN-P2 (rang 1)        : {len(sirens_v_p2):,}")

# SN-P3 : feuille Top3, idem rang 1 validé
df_p3 = pd.read_excel(SN_PHASE3, sheet_name='Top3', dtype=str)
df_p3['rang'] = pd.to_numeric(df_p3['rang'], errors='coerce')
r1_p3 = df_p3[df_p3['rang'] == 1]
v_p3 = r1_p3[r1_p3['statut_candidat'].isin(['VALIDE_FORT', 'VALIDE'])]
sirens_v_p3 = {_norm_siren(v) for v in v_p3.get('siren_ref_app', pd.Series(dtype=str)).dropna()}
sirens_v_p3.discard("")
sirens_cibles |= sirens_v_p3
print(f"SIREN validés en SN-P3 (rang 1)        : {len(sirens_v_p3):,}")

# Source 2 : SIREN candidats top 5 en SN-P2 (peu importe le statut)
sirens_p2_all = {_norm_siren(v) for v in df_p2.get('siren_ref', pd.Series(dtype=str)).dropna()}
sirens_p2_all.discard("")
sirens_cibles |= sirens_p2_all
print(f"SIREN candidats top 5 en SN-P2 (tous)  : {len(sirens_p2_all):,}")

# Source 3 : SIREN candidats top 3 en SN-P3 (peu importe le statut)
sirens_p3_all = {_norm_siren(v) for v in df_p3.get('siren_ref_app', pd.Series(dtype=str)).dropna()}
sirens_p3_all.discard("")
sirens_cibles |= sirens_p3_all
print(f"SIREN candidats top 3 en SN-P3 (tous)  : {len(sirens_p3_all):,}")

print(f"\nTotal SIREN cibles uniques : {len(sirens_cibles):,}")

SIREN validés en SN-P1 : 38,154
SIREN validés en SN-P2 (rang 1)        : 12,524
SIREN validés en SN-P3 (rang 1)        : 973
SIREN candidats top 5 en SN-P2 (tous)  : 75,767
SIREN candidats top 3 en SN-P3 (tous)  : 8,921

Total SIREN cibles uniques : 116,079


## 2. Filtrage de la base SIRENE Etab

In [3]:
df_etab = pd.read_parquet(SIRENE_ETAB_CLEAN)
print(f"Etab SIRENE complète : {len(df_etab):,}")

df_etab['siren'] = df_etab['siren'].astype(str)
df_etab_filtre = df_etab[df_etab['siren'].isin(sirens_cibles)].copy().reset_index(drop=True)

print(f"Etab SIRENE filtrée  : {len(df_etab_filtre):,}")
print(f"Réduction            : {(1 - len(df_etab_filtre)/len(df_etab))*100:.1f}%")
print(f"SIREN distincts      : {df_etab_filtre['siren'].nunique():,}")
print(f"  (parmi {len(sirens_cibles):,} SIREN cibles, "
      f"{(df_etab_filtre['siren'].nunique()/len(sirens_cibles))*100:.1f}% ont au moins un Etab actif)")

Etab SIRENE complète : 16,867,946
Etab SIRENE filtrée  : 244,841
Réduction            : 98.5%
SIREN distincts      : 116,079
  (parmi 116,079 SIREN cibles, 100.0% ont au moins un Etab actif)


## 3. Aperçu

In [4]:
afficher_tableau(
    df_etab_filtre[['siret', 'siren', 'denominationUniteLegale',
                    'enseigne1Etablissement', 'adresse_complete_etab',
                    'codeCommuneEtablissement', 'dept_etab']],
    f"Aperçu Etab SIRENE filtré ({len(df_etab_filtre):,} lignes)",
)

siret,siren,denominationUniteLegale,enseigne1Etablissement,adresse_complete_etab,codeCommuneEtablissement,dept_etab
00572016400028,005720164,SAINTE ISABELLE,None,236 ROUTE D'AMIENS,80001,80
00578005100024,005780051,PHARMACIE CARE,None,9 CHEMIN DE CHEMOULIN,44184,44
00705004000018,007050040,CENTRE DES CARMES,None,689 AVENUE MARIUS AUTRIC,04001,04
00705007300019,007050073,CENTRE REEDAPTAT L EAU VIVE,None,522 ROUTE DES GARCINETS,04222,04
00715017000038,007150170,ETABLISSEMENTS BIBAL,None,3 CHEMIN DES AIRES,04049,04


## 4. Sauvegarde

In [5]:
df_etab_filtre.to_parquet(SIRENE_ETAB_FILTRE, index=False)
print(f"Sauvegardé : {SIRENE_ETAB_FILTRE}")

Sauvegardé : /home/jovyan/work/projet_finess_sirene/data/processed/sirene_etab_filtre.parquet
